In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

I0000 00:00:1788544801.093496   57966 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788544801.337045   57966 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788544803.342967   57966 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
train_path = "/mnt/d/Medical-Image-Diagnosis/data/bone_fracture/train"
validation_path = "/mnt/d/Medical-Image-Diagnosis/data/bone_fracture/val"
test_path = "/mnt/d/Medical-Image-Diagnosis/data/bone_fracture/test"

In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)

training_set = train_datagen.flow_from_directory(
    train_path,
    target_size=(224, 224),
    color_mode="rgb",
    batch_size=32,
    class_mode="binary",
    shuffle=True
)

validation_set = test_datagen.flow_from_directory(
    validation_path,
    target_size=(224, 224),
    color_mode="rgb",
    batch_size=32,
    class_mode="binary",
    shuffle=False
)

test_set = test_datagen.flow_from_directory(
    test_path,
    target_size=(224, 224),
    color_mode="rgb",
    batch_size=32,
    class_mode="binary",
    shuffle=False
)

Found 9240 images belonging to 2 classes.
Found 823 images belonging to 2 classes.
Found 500 images belonging to 2 classes.


In [4]:
print("Training class indices:", training_set.class_indices)
print("Validation class indices:", validation_set.class_indices)
print("Test class indices:", test_set.class_indices)

print("Training samples:", training_set.samples)
print("Validation samples:", validation_set.samples)
print("Test samples:", test_set.samples)

print("Image shape:", training_set.image_shape)

Training class indices: {'fractured': 0, 'not fractured': 1}
Validation class indices: {'fractured': 0, 'not fractured': 1}
Test class indices: {'fractured': 0, 'not fractured': 1}
Training samples: 9240
Validation samples: 823
Test samples: 500
Image shape: (224, 224, 3)


In [5]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

I0000 00:00:1788544978.530284   57966 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3536 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 17s 2us/step


In [6]:
model_tl = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

model_tl.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [8]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-7
)

In [9]:
model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [10]:
history_tl = model_tl.fit(
    training_set,
    validation_data=validation_set,
    epochs=20,
    callbacks=[early_stopping, reduce_lr]
)

Epoch 1/20


I0000 00:00:1788545113.517654   57966 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1788545116.270023   58638 service.cc:153] XLA service 0x7d7d8c035370 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1788545116.270050   58638 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9 (Driver: 13.4.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.25.1)
I0000 00:00:1788545116.339005   58638 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1788545116.970091   58638 cuda_dnn.cc:461] Loaded cuDNN version 92501
I0000 00:00:1788545117.032157   58638 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8125__.115
E0000 00:00:1788545119.459399   58638 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup e

  2/289 ━━━━━━━━━━━━━━━━━━━━ 16s 56ms/step - accuracy: 0.5000 - loss: 0.8241 

I0000 00:00:1788545125.567365   58638 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 48/289 ━━━━━━━━━━━━━━━━━━━━ 1:24 351ms/step - accuracy: 0.6016 - loss: 0.6871

I0000 00:00:1788545142.795325   58641 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8125__.115
E0000 00:00:1788545143.859578   58641 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


289/289 ━━━━━━━━━━━━━━━━━━━━ 0s 377ms/step - accuracy: 0.7301 - loss: 0.5341

/home/srijan_yadav07/medical-env/lib/python3.12/site-packages/PIL/Image.py:1136: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
E0000 00:00:1788545249.851221   58638 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


289/289 ━━━━━━━━━━━━━━━━━━━━ 138s 438ms/step - accuracy: 0.7301 - loss: 0.5341 - val_accuracy: 0.8323 - val_loss: 0.4094 - learning_rate: 1.0000e-04
Epoch 2/20
289/289 ━━━━━━━━━━━━━━━━━━━━ 114s 395ms/step - accuracy: 0.8440 - loss: 0.3707 - val_accuracy: 0.8615 - val_loss: 0.3264 - learning_rate: 1.0000e-04
Epoch 3/20
289/289 ━━━━━━━━━━━━━━━━━━━━ 122s 424ms/step - accuracy: 0.8811 - loss: 0.3019 - val_accuracy: 0.8761 - val_loss: 0.2851 - learning_rate: 1.0000e-04
Epoch 4/20
289/289 ━━━━━━━━━━━━━━━━━━━━ 250s 865ms/step - accuracy: 0.9042 - loss: 0.2523 - val_accuracy: 0.8991 - val_loss: 0.2506 - learning_rate: 1.0000e-04
Epoch 5/20
289/289 ━━━━━━━━━━━━━━━━━━━━ 123s 426ms/step - accuracy: 0.9269 - loss: 0.2101 - val_accuracy: 0.9149 - val_loss: 0.2314 - learning_rate: 1.0000e-04
Epoch 6/20
289/289 ━━━━━━━━━━━━━━━━━━━━ 132s 457ms/step - accuracy: 0.9364 - loss: 0.1846 - val_accuracy: 0.9271 - val_loss: 0.2047 - learning_rate: 1.0000e-04
Epoch 7/20
289/289 ━━━━━━━━━━━━━━━━━━━━ 149s 516ms/